# 02 Train 2D CNN v0.9

**Purpose:** UI-only tmux control panel with a clear default path for training a *new* model directory, plus a separate mode for resuming an existing model trajectory and a compact structured epoch summary panel.  
**Inputs:** `./src/train.py`, `./src/v0.2/training_control_panel.py`, `./src/resume/control_panel.py`, existing model run artifacts under `./models/`.  
**Expected outputs:** canonical run artifacts under `./models/<model_directory>/runs/<run_id>/` with `train.log`, `best.pt`, `latest.pt`, and metadata files.  
**Artifact write mode:** canonical artifacts are written by `src/train.py`; this notebook orchestrates launch/session/log controls.  
**Decision supported:** `READY_FOR_LAUNCH` vs `PATCH_REQUIRED`


In [1]:
# Repo Setup
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    for parent in repo_root.parents:
        if (parent / 'src').exists() and (parent / 'training-data').exists() and (parent / 'validation-data').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

helper_root = (repo_root / 'src' / 'v0.2').resolve()
if str(helper_root) not in sys.path:
    sys.path.insert(0, str(helper_root))

print(f'repo_root={repo_root}')
print(f'helper_root={helper_root}')


repo_root=/home/mitch/development/raccoon-ball/03_rb-training-v2.0
helper_root=/home/mitch/development/raccoon-ball/03_rb-training-v2.0/src/v0.2


In [2]:
# Panel Config
REPO_ROOT = repo_root
PYTHON_EXECUTABLE = sys.executable
TRAINING_MODULE = 'src.train'

TRAINING_DATA_ROOT = REPO_ROOT / 'training-data'
VALIDATION_DATA_ROOT = REPO_ROOT / 'validation-data'
MODELS_ROOT = REPO_ROOT / 'models'

MODEL_SUFFIX_DEFAULT = 'ds-2d-cnn'
TOPOLOGY_ID_DEFAULT = 'distance_regressor_dual_stream_yaw'
MODEL_NAME_DEFAULT = 'dual_stream_yaw_v0_1'
TOPOLOGY_PARAMS_PRESETS = {
    'distance_regressor_dual_stream': '{\"output_mode\":\"scalar_distance\",\"output_dim\":1,\"dropout_p\":0.0}',
    'distance_regressor_dual_stream_yaw': '{\"dropout_p\":0.0}',
    'distance_regressor_tri_stream_yaw': '{\"dropout_p\":0.0}',
    'distance_regressor_2d_cnn': '{\"dropout_p\":0.2,\"input_channels\":1}',
    'distance_regressor_global_pool_cnn': '{\"dropout_p\":0.2,\"input_channels\":1}',
}
TOPOLOGY_PARAMS_JSON_DEFAULT = TOPOLOGY_PARAMS_PRESETS.get(TOPOLOGY_ID_DEFAULT, '{}')
LOG_FILENAME = 'train.log'
DEFAULT_LOG_TAIL_LINES = 180
DEFAULT_LOG_POLL_SECONDS = 5.0
DEFAULT_ADDITIONAL_EPOCHS = 0
WORKFLOW_NEW = 'new_model'
WORKFLOW_RESUME = 'resume_existing'
RESERVED_PARENT_RUN_ID = '[reserved for future implementation]]'

TRAIN_LAUNCH_DEFAULTS = {
    'seed': 42,
    'batch_size': 8,
    'epochs': 32,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'early_stopping_patience': 3,
    'change_note': 'tmux control-panel launch via notebook v0.9',
}

print(f'python={PYTHON_EXECUTABLE}')
print(f'train_module={TRAINING_MODULE}')
print(f'models_root={MODELS_ROOT}')


python=/home/mitch/development/raccoon-ball/.venv/bin/python
train_module=src.train
models_root=/home/mitch/development/raccoon-ball/03_rb-training-v2.0/models


In [3]:
# Helper Imports
import ast
import html
import json
import re
import threading

import ipywidgets as widgets
from IPython.display import display

import training_control_panel as control
from src.resume import control_panel as resume_control
from src.resume.state import RESUME_STATE_FILENAME, load_resume_state
from src.topologies import get_topology_definition, list_topology_ids

print(f'loaded_training_helper={control.__file__}')
print(f'loaded_resume_helper={resume_control.__file__}')


loaded_training_helper=/home/mitch/development/raccoon-ball/03_rb-training-v2.0/src/v0.2/training_control_panel.py
loaded_resume_helper=/home/mitch/development/raccoon-ball/03_rb-training-v2.0/src/resume/control_panel.py


In [4]:
# Control Panel
_RUN_ID_RE = re.compile(r'^run_([0-9]+)$')

status_area = widgets.HTML(value='')
workflow_help = widgets.HTML(value='')
session_suggestion = widgets.HTML(value='')

workflow_toggle = widgets.ToggleButtons(
    options=[
        ('Train New Model', WORKFLOW_NEW),
        ('Resume Existing Model', WORKFLOW_RESUME),
    ],
    value=WORKFLOW_NEW,
    description='Workflow',
)
prepare_new_model_button = widgets.Button(
    description='Prepare New Model Directory',
    button_style='success',
)

topology_id_select = widgets.Dropdown(
    options=(TOPOLOGY_ID_DEFAULT,),
    value=TOPOLOGY_ID_DEFAULT,
    description='Topology ID',
    layout=widgets.Layout(width='420px'),
)
topology_variant_select = widgets.Dropdown(
    options=(MODEL_NAME_DEFAULT,),
    value=MODEL_NAME_DEFAULT,
    description='Top. Variant',
    layout=widgets.Layout(width='320px'),
)
refresh_topologies_button = widgets.Button(description='Refresh Topologies')
topology_note_input = widgets.Text(
    value='',
    description='Top. Note',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)
topology_help = widgets.HTML(value='')
model_select = widgets.Dropdown(
    options=[],
    value=None,
    description='Model Select',
    layout=widgets.Layout(width='420px'),
)
refresh_models_button = widgets.Button(description='Refresh Models')
get_info_button = widgets.Button(description='Get Info', button_style='info')
model_directory_input = widgets.Text(
    value='',
    description='Model Dir',
    layout=widgets.Layout(width='420px'),
)

session_name_input = widgets.Text(
    value='',
    description='Session Name',
    placeholder='auto-generated for selected workflow',
    layout=widgets.Layout(width='420px'),
)

seed_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['seed'], description='Seed')
batch_size_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['batch_size'], description='Batch Size')
epochs_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['epochs'], description='Epochs')
additional_epochs_input = widgets.IntText(
    value=DEFAULT_ADDITIONAL_EPOCHS,
    description='Add Epochs',
    layout=widgets.Layout(width='220px'),
)
learning_rate_input = widgets.FloatText(value=TRAIN_LAUNCH_DEFAULTS['learning_rate'], description='LR')
weight_decay_input = widgets.FloatText(value=TRAIN_LAUNCH_DEFAULTS['weight_decay'], description='Weight Decay')
early_stop_input = widgets.IntText(value=TRAIN_LAUNCH_DEFAULTS['early_stopping_patience'], description='Early Stop')
enable_lr_scheduler_toggle = widgets.Checkbox(
    value=True,
    description='LR Scheduler',
    indent=False,
    layout=widgets.Layout(width='180px'),
)
change_note_input = widgets.Text(
    value=TRAIN_LAUNCH_DEFAULTS['change_note'],
    description='Change Note',
    layout=widgets.Layout(width='760px'),
)
topology_params_json_input = widgets.Text(
    value=TOPOLOGY_PARAMS_JSON_DEFAULT,
    description='Top. Params',
    layout=widgets.Layout(width='760px'),
)
primary_variable_input = widgets.Text(
    value='',
    description='Primary Var',
    placeholder='e.g. bbox line width 3px -> 1px',
    layout=widgets.Layout(width='760px'),
)
notes_input = widgets.Text(
    value='',
    description='Notes',
    layout=widgets.Layout(width='760px'),
)

run_id_preview = widgets.Text(
    value='',
    description='Run ID',
    disabled=True,
    layout=widgets.Layout(width='320px'),
)
source_run_preview = widgets.Text(
    value='',
    description='Source Run',
    disabled=True,
    layout=widgets.Layout(width='320px'),
)
run_register_path = widgets.Text(
    value='',
    description='Run Register',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)
derived_log_path = widgets.Text(
    value='',
    description='Log Path',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)

launch_plan_output = widgets.Textarea(
    value='',
    description='Launch Plan',
    layout=widgets.Layout(width='100%', height='130px'),
    disabled=True,
)
model_info_output = widgets.Textarea(
    value='',
    description='Model Info',
    layout=widgets.Layout(width='100%', height='250px'),
    disabled=True,
)

session_select = widgets.Dropdown(
    options=[],
    value=None,
    description='Sessions',
    layout=widgets.Layout(width='420px'),
)

refresh_sessions_button = widgets.Button(description='Refresh Sessions')
launch_button = widgets.Button(description='Launch', button_style='success')
end_session_button = widgets.Button(description='End Session', button_style='danger')

log_tail_lines_input = widgets.IntText(
    value=DEFAULT_LOG_TAIL_LINES,
    description='Tail Lines',
    layout=widgets.Layout(width='220px'),
)
poll_interval_input = widgets.FloatText(
    value=DEFAULT_LOG_POLL_SECONDS,
    description='Poll Secs',
    layout=widgets.Layout(width='220px'),
)
auto_refresh_toggle = widgets.ToggleButton(
    value=False,
    description='Auto Refresh',
    icon='refresh',
    layout=widgets.Layout(width='180px'),
)
refresh_log_button = widgets.Button(description='Refresh Log', button_style='info')

epoch_summary_output = widgets.HTML(value='')

log_output = widgets.Textarea(
    value='',
    description='',
    layout=widgets.Layout(width='100%', height='420px'),
    disabled=True,
)

_auto_refresh_state = {'thread': None, 'stop_event': None}
_session_name_state = {'last_auto': ''}
_topology_params_state = {'last_auto': TOPOLOGY_PARAMS_JSON_DEFAULT}
_topology_metadata_state = {'by_topology_id': {}, 'scan_error': None}
_model_state = {
    'rows': [],
    'infos': [],
    'by_run_id': {},
    'latest_resumable': None,
}


def _run_sort_index(run_id: str) -> int:
    match = _RUN_ID_RE.fullmatch(str(run_id).strip())
    if match:
        return int(match.group(1))
    return -1


def _set_status(message: str, *, is_error: bool = False) -> None:
    color = '#9b1c1c' if is_error else '#0f5132'
    lr_scheduler_text = 'enabled' if bool(enable_lr_scheduler_toggle.value) else 'disabled'
    status_area.value = (
        f"<div style='border:1px solid #d0d7de;padding:8px;border-radius:6px;'>"
        f"<strong style='color:{color};'>Status</strong>"
        f"<div><code>lr_scheduler={lr_scheduler_text}</code></div>"
        f"<div>{html.escape(message)}</div>"
        '</div>'
    )


def _set_epoch_summary_text(message: str) -> None:
    epoch_summary_output.value = (
        "<div style='border:1px solid #d0d7de;padding:8px;border-radius:6px;'>"
        "<strong>Epoch Summary</strong>"
        f"<pre style='white-space:pre-wrap;margin:8px 0 0 0;font-family:Menlo,Consolas,monospace;font-size:12px;line-height:1.35;'>"
        f"{html.escape(message)}"
        "</pre>"
        "</div>"
    )


def _sync_default_session_name(suggested_session: str) -> None:
    current = session_name_input.value.strip()
    last_auto = _session_name_state['last_auto']
    if (not current) or (current == last_auto):
        session_name_input.value = suggested_session
    _session_name_state['last_auto'] = suggested_session


def _set_workflow_help() -> None:
    mode = workflow_toggle.value
    if mode == WORKFLOW_NEW:
        workflow_help.value = (
            '<div style="border:1px solid #d0d7de;padding:8px;border-radius:6px;">'
            '<strong>New Model Path</strong>'
            '<ol style="margin:6px 0 0 20px;">'
            '<li>Click <code>Prepare New Model Directory</code>.</li>'
            '<li>Verify <code>Model Dir</code> is a fresh timestamped value.</li>'
            '<li>Set hyperparameters and click <code>Launch</code>.</li>'
            '</ol>'
            '</div>'
        )
    else:
        workflow_help.value = (
            '<div style="border:1px solid #d0d7de;padding:8px;border-radius:6px;">'
            '<strong>Resume Existing Path</strong>'
            '<ol style="margin:6px 0 0 20px;">'
            '<li>Select an existing model in <code>Model Select</code>.</li>'
            '<li>Click <code>Get Info</code> to confirm completed/total epochs.</li>'
            '<li>Set optional <code>Add Epochs</code> and click <code>Launch</code>.</li>'
            '</ol>'
            '</div>'
        )


def _extract_topology_literals(module_path: Path) -> dict[str, object]:
    names = {"TOPOLOGY_ID", "DEFAULT_VARIANT", "TOPOLOGY_METADATA"}
    source = module_path.read_text(encoding='utf-8')
    tree = ast.parse(source, filename=str(module_path))

    values: dict[str, object] = {}
    for node in tree.body:
        target_name = None
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name):
                target_name = target.id
                value_node = node.value
            else:
                continue
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
            target_name = node.target.id
            value_node = node.value
        else:
            continue

        if target_name not in names or value_node is None:
            continue
        try:
            values[target_name] = ast.literal_eval(value_node)
        except Exception:
            continue
    return values


def _scan_topology_metadata_files() -> None:
    topology_dir = (REPO_ROOT / 'src' / 'topologies').resolve()
    by_topology_id: dict[str, list[dict[str, str]]] = {}
    errors: list[str] = []

    if not topology_dir.exists():
        _topology_metadata_state['by_topology_id'] = {}
        _topology_metadata_state['scan_error'] = f'missing topology directory: {topology_dir}'
        return

    for module_path in sorted(topology_dir.glob('topology_*.py')):
        try:
            literals = _extract_topology_literals(module_path)
        except Exception as exc:
            errors.append(f'{module_path.name}: {exc}')
            continue

        topology_id = str(literals.get('TOPOLOGY_ID', '')).strip()
        if not topology_id:
            continue

        default_variant = str(literals.get('DEFAULT_VARIANT', '')).strip()
        raw_metadata = literals.get('TOPOLOGY_METADATA')
        metadata = raw_metadata if isinstance(raw_metadata, dict) else {}

        status = str(metadata.get('status', 'active')).strip().lower() or 'active'
        if status not in {'active', 'experimental', 'deprecated'}:
            status = 'active'

        entry = {
            'status': status,
            'display_name': str(metadata.get('display_name', '')).strip(),
            'note': str(metadata.get('note', '')).strip(),
            'replacement': str(metadata.get('replacement', '')).strip(),
            'default_variant': default_variant,
            'source_file': module_path.name,
        }
        by_topology_id.setdefault(topology_id, []).append(entry)

    _topology_metadata_state['by_topology_id'] = by_topology_id
    _topology_metadata_state['scan_error'] = '; '.join(errors) if errors else None


def _resolve_topology_metadata(topology_id: str) -> dict[str, str]:
    topology_key = str(topology_id).strip()
    fallback = {
        'status': 'active',
        'display_name': topology_key,
        'note': '',
        'replacement': '',
        'default_variant': '',
        'source_file': '',
    }
    if not topology_key:
        return dict(fallback)

    entries = _topology_metadata_state['by_topology_id'].get(topology_key) or []
    if not entries:
        return dict(fallback)

    registry_default_variant = ''
    try:
        registry_default_variant = str(get_topology_definition(topology_key).default_variant).strip()
    except Exception:
        registry_default_variant = ''

    selected_entry = None
    if registry_default_variant:
        selected_entry = next(
            (entry for entry in entries if str(entry.get('default_variant', '')).strip() == registry_default_variant),
            None,
        )
    if selected_entry is None:
        selected_entry = entries[0]

    merged = dict(fallback)
    merged.update({k: str(v).strip() for k, v in selected_entry.items()})
    status = str(merged.get('status', 'active')).strip().lower() or 'active'
    merged['status'] = status if status in {'active', 'experimental', 'deprecated'} else 'active'
    if not merged.get('display_name'):
        merged['display_name'] = topology_key
    return merged


def _suggest_topology_params_json(topology_id: str) -> str:
    selected_id = str(topology_id).strip()
    preset = TOPOLOGY_PARAMS_PRESETS.get(selected_id)
    if isinstance(preset, str) and preset.strip():
        return preset
    return '{}'


def _sync_default_topology_params(topology_id: str) -> None:
    suggested = _suggest_topology_params_json(topology_id)
    current = topology_params_json_input.value.strip()
    last_auto = _topology_params_state['last_auto']
    if (not current) or (current == last_auto):
        topology_params_json_input.value = suggested
    _topology_params_state['last_auto'] = suggested


def _sync_topology_note(topology_id: str) -> None:
    if not str(topology_id).strip():
        topology_note_input.value = ''
        return

    metadata = _resolve_topology_metadata(topology_id)
    note = str(metadata.get('note', '')).strip()
    status = str(metadata.get('status', 'active')).strip().lower() or 'active'
    replacement = str(metadata.get('replacement', '')).strip()
    if status == 'deprecated' and replacement:
        suffix = f'Replacement: {replacement}'
        note = f'{note} {suffix}'.strip() if note else suffix
    topology_note_input.value = note


def _set_topology_help() -> None:
    topology_id = str(topology_id_select.value or '').strip()
    topology_variant = str(topology_variant_select.value or '').strip()
    _sync_topology_note(topology_id)

    if not topology_id:
        topology_help.value = (
            '<div style="border:1px solid #d0d7de;padding:8px;border-radius:6px;">'
            '<strong>Topology Selection</strong>'
            '<div>Pick a topology family and variant before launch.</div>'
            '</div>'
        )
        return

    metadata = _resolve_topology_metadata(topology_id)
    try:
        definition = get_topology_definition(topology_id)
        variants = ', '.join(definition.supported_variants)
        replacement_text = str(metadata.get('replacement', '')).strip()
        replacement_row = (
            f'<div>Replacement: <code>{html.escape(replacement_text)}</code></div>'
            if replacement_text
            else ''
        )
        topology_help.value = (
            '<div style="border:1px solid #d0d7de;padding:8px;border-radius:6px;">'
            '<strong>Topology Selection</strong>'
            f"<div>Display name: <code>{html.escape(str(metadata.get('display_name', topology_id)))}</code></div>"
            f"<div>Status: <code>{html.escape(str(metadata.get('status', 'active')))}</code></div>"
            f'<div><code>Topology ID</code> family: <code>{html.escape(topology_id)}</code></div>'
            f'<div><code>Top. Variant</code> architecture: <code>{html.escape(topology_variant or definition.default_variant)}</code></div>'
            f'<div>Model class: <code>{html.escape(definition.model_class_name)}</code></div>'
            f'<div>Supported variants: <code>{html.escape(variants)}</code></div>'
            f'{replacement_row}'
            '</div>'
        )
    except Exception as exc:
        topology_help.value = (
            '<div style="border:1px solid #f5c2c7;padding:8px;border-radius:6px;">'
            '<strong>Topology Selection</strong>'
            f'<div>Registry lookup failed: {html.escape(str(exc))}</div>'
            '</div>'
        )


def _sync_topology_variant_options(*, preferred_variant: str | None = None) -> None:
    topology_id = str(topology_id_select.value or '').strip() or TOPOLOGY_ID_DEFAULT
    current_variant = str(topology_variant_select.value or '').strip()
    preferred = str(preferred_variant or '').strip()
    options: tuple[str, ...] = ()
    default_variant = MODEL_NAME_DEFAULT

    try:
        definition = get_topology_definition(topology_id)
        options = tuple(definition.supported_variants)
        default_variant = str(definition.default_variant).strip() or MODEL_NAME_DEFAULT
    except Exception:
        options = tuple()

    if not options:
        fallback = preferred or current_variant or default_variant
        options = (fallback,)

    selected = preferred if preferred in options else None
    if selected is None and current_variant in options:
        selected = current_variant
    if selected is None and default_variant in options:
        selected = default_variant
    if selected is None:
        selected = options[0]

    topology_variant_select.options = options
    topology_variant_select.value = selected
    _sync_default_topology_params(topology_id)
    _set_topology_help()


def _refresh_topology_options(*, update_status: bool = False) -> None:
    current_topology_id = str(topology_id_select.value or '').strip()
    try:
        topology_ids = tuple(list_topology_ids(include_deprecated=False))
        if not topology_ids:
            raise ValueError('Topology registry returned no topology ids.')
    except Exception as exc:
        topology_id_select.options = ((TOPOLOGY_ID_DEFAULT, TOPOLOGY_ID_DEFAULT),)
        topology_id_select.value = TOPOLOGY_ID_DEFAULT
        _sync_topology_variant_options(preferred_variant=MODEL_NAME_DEFAULT)
        if update_status:
            _set_status(f'Topology registry refresh failed: {exc}', is_error=True)
        return

    _scan_topology_metadata_files()

    ordered_statuses = ('active', 'experimental')
    topology_options_by_status: dict[str, list[tuple[str, str]]] = {
        status: [] for status in ordered_statuses
    }
    selectable_topology_ids: list[str] = []

    for topology_id in topology_ids:
        metadata = _resolve_topology_metadata(topology_id)
        status = str(metadata.get('status', 'active')).strip().lower() or 'active'
        if status not in topology_options_by_status:
            status = 'active'
        display_name = str(metadata.get('display_name', '')).strip() or topology_id
        option_label = f'{display_name} [{status}]'
        topology_options_by_status[status].append((option_label, topology_id))
        selectable_topology_ids.append(topology_id)

    ordered_options: list[tuple[str, str]] = []
    for status in ordered_statuses:
        ordered_options.extend(topology_options_by_status[status])

    if not ordered_options:
        ordered_options = [(topology_id, topology_id) for topology_id in topology_ids]
        selectable_topology_ids = list(topology_ids)

    topology_id_select.options = tuple(ordered_options)
    if current_topology_id in selectable_topology_ids:
        topology_id_select.value = current_topology_id
    elif TOPOLOGY_ID_DEFAULT in selectable_topology_ids:
        topology_id_select.value = TOPOLOGY_ID_DEFAULT
    else:
        topology_id_select.value = selectable_topology_ids[0]

    _sync_topology_variant_options()
    if update_status:
        scan_error = _topology_metadata_state.get('scan_error')
        if scan_error:
            _set_status(f'Topology registry refreshed (metadata scan warnings: {scan_error}).')
        else:
            _set_status('Topology registry refreshed.')


def _suggest_unique_model_directory() -> str:
    existing = set(control.list_model_directories(MODELS_ROOT))
    base = control.suggest_model_directory(MODELS_ROOT, model_suffix=MODEL_SUFFIX_DEFAULT)
    if base not in existing:
        return base
    for index in range(2, 200):
        candidate = f'{base}-v{index:02d}'
        if candidate not in existing:
            return candidate
    raise RuntimeError('Could not suggest a unique model directory name.')


def _load_model_run_rows(model_directory: str) -> tuple[list[dict], Path]:
    model_dir = control.build_model_dir(models_root=MODELS_ROOT, model_directory=model_directory)
    register_path = model_dir / 'run_register.json'
    if not register_path.exists():
        return [], register_path

    with register_path.open('r', encoding='utf-8') as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f'run_register.json must be an object: {register_path}')

    raw_runs = payload.get('runs', [])
    if raw_runs is None:
        raw_runs = []
    if not isinstance(raw_runs, list):
        raise ValueError(f"run_register.json 'runs' must be a list: {register_path}")

    rows: list[dict] = []
    for row in raw_runs:
        if not isinstance(row, dict):
            continue
        run_id = str(row.get('run_id', '')).strip()
        if not _RUN_ID_RE.fullmatch(run_id):
            continue
        rows.append(dict(row))

    rows.sort(key=lambda row: _run_sort_index(str(row.get('run_id', ''))))
    return rows, register_path


def _inspect_run(model_directory: str, row: dict) -> dict:
    run_id = str(row.get('run_id', '')).strip()
    run_dir = control.build_run_dir(
        run_id=run_id,
        models_root=MODELS_ROOT,
        model_directory=model_directory,
    ).resolve()

    info: dict = {
        'run_id': run_id,
        'run_dir': str(run_dir),
        'exists': run_dir.exists(),
        'total_epochs': None,
        'completed_epochs': None,
        'remaining_epochs': None,
        'is_complete': None,
        'is_resumable': False,
        'state_error': None,
        'topology_id': str(row.get('topology_id', '')).strip() or str(topology_id_select.value or '').strip() or TOPOLOGY_ID_DEFAULT,
        'topology_variant': str(row.get('topology_variant', '')).strip() or str(row.get('model_name', '')).strip() or str(topology_variant_select.value or '').strip(),
        'architecture_variant': str(row.get('model_name', '')).strip() or str(topology_variant_select.value or '').strip(),
    }

    config_path = run_dir / 'config.json'
    if config_path.exists():
        try:
            with config_path.open('r', encoding='utf-8') as handle:
                config_payload = json.load(handle)
            total_epochs = config_payload.get('epochs')
            if total_epochs is not None:
                info['total_epochs'] = int(total_epochs)

            topology_id = str(config_payload.get('topology_id', '')).strip()
            if topology_id:
                info['topology_id'] = topology_id

            topology_variant = str(config_payload.get('topology_variant', '')).strip()
            if not topology_variant:
                topology_variant = str(config_payload.get('model_architecture_variant', '')).strip()
            if topology_variant:
                info['topology_variant'] = topology_variant
                info['architecture_variant'] = topology_variant
        except Exception as exc:
            info['state_error'] = f'config read failed: {exc}'

    resume_state_path = run_dir / RESUME_STATE_FILENAME
    if resume_state_path.exists():
        try:
            resume_state = load_resume_state(resume_state_path, map_location='cpu')
            info['completed_epochs'] = int(resume_state.get('epoch'))
            info['is_resumable'] = True
        except Exception as exc:
            info['state_error'] = f'resume state read failed: {exc}'

    total = info['total_epochs']
    completed = info['completed_epochs']
    if isinstance(total, int) and isinstance(completed, int):
        remaining = max(total - completed, 0)
        info['remaining_epochs'] = remaining
        info['is_complete'] = remaining == 0

    return info


def _refresh_models(*_) -> None:
    try:
        model_dirs = control.list_model_directories(MODELS_ROOT)
    except Exception as exc:
        model_select.options = []
        model_select.value = None
        _set_status(f'Model directory refresh failed: {exc}', is_error=True)
        return

    preferred = model_directory_input.value.strip()
    model_select.options = model_dirs

    if model_dirs:
        if preferred in model_dirs:
            model_select.value = preferred
        else:
            model_select.value = model_dirs[-1]
    else:
        model_select.value = None


def _refresh_model_state(*, show_error: bool = False) -> None:
    model_directory = model_directory_input.value.strip()
    _model_state['rows'] = []
    _model_state['infos'] = []
    _model_state['by_run_id'] = {}
    _model_state['latest_resumable'] = None

    source_run_preview.value = ''
    if not model_directory:
        run_register_path.value = ''
        return

    try:
        rows, register_path = _load_model_run_rows(model_directory)
    except Exception as exc:
        run_register_path.value = ''
        if show_error:
            _set_status(f'Run state refresh failed: {exc}', is_error=True)
        return

    run_register_path.value = str(register_path)
    infos = [_inspect_run(model_directory, row) for row in rows]
    infos.sort(key=lambda item: _run_sort_index(str(item.get('run_id', ''))))
    by_run_id = {str(item['run_id']): item for item in infos}

    resumable_infos = [item for item in infos if item.get('is_resumable')]
    latest_resumable = None
    if resumable_infos:
        latest_resumable = sorted(
            resumable_infos,
            key=lambda item: _run_sort_index(str(item.get('run_id', ''))),
        )[-1]

    _model_state['rows'] = rows
    _model_state['infos'] = infos
    _model_state['by_run_id'] = by_run_id
    _model_state['latest_resumable'] = latest_resumable

    if latest_resumable is not None:
        source_run_preview.value = str(latest_resumable['run_id'])


def _build_model_info_text(model_directory: str) -> str:
    lines: list[str] = [f'model_directory: {model_directory}', f'workflow: {workflow_toggle.value}', '']
    infos = list(_model_state['infos'])
    if not infos:
        lines.append('No registered runs found for this model directory.')
        return "\n".join(lines)

    latest = _model_state['latest_resumable']
    if latest is not None:
        completed = latest.get('completed_epochs')
        total = latest.get('total_epochs')
        lines.append(
            f"latest_resumable: {latest['run_id']} "
            f"(completed={completed if completed is not None else '?'} / "
            f"total={total if total is not None else '?'})"
        )
    else:
        lines.append('latest_resumable: none')

    lines.append(f'registered_runs: {len(infos)}')
    lines.append('')
    lines.append('Run details (completed / total):')

    for info in infos:
        completed = info.get('completed_epochs')
        total = info.get('total_epochs')
        remaining = info.get('remaining_epochs')
        completed_text = '?' if completed is None else str(int(completed))
        total_text = '?' if total is None else str(int(total))
        remaining_text = '?' if remaining is None else str(int(remaining))
        resume_text = 'yes' if info.get('is_resumable') else 'no'
        topology_id_text = str(info.get('topology_id') or '?')
        topology_variant_text = str(
            info.get('topology_variant') or info.get('architecture_variant') or '?'
        )
        lines.append(
            f"- {info['run_id']}: topology={topology_id_text}/{topology_variant_text}; "
            f"completed={completed_text} / total={total_text}; "
            f"remaining={remaining_text}; resumable={resume_text}"
        )
        state_error = info.get('state_error')
        if state_error:
            lines.append(f'  warning: {state_error}')

    return "\n".join(lines)


def _build_launch_plan_text(preview: dict | None) -> str:
    if preview is None:
        return '[launch plan unavailable]'

    mode = preview.get('mode')
    model_directory = preview.get('model_directory')
    run_id = preview.get('run_id')
    lines = [
        f'mode={mode}',
        f'model_directory={model_directory}',
        f'target_run_id={run_id}',
        f"topology_id={preview.get('topology_id')}",
        f"topology_variant={preview.get('topology_variant')}",
    ]

    if mode == WORKFLOW_NEW:
        existing_rows = int(preview.get('existing_rows', 0))
        if existing_rows > 0:
            lines.append('action=blocked (model directory already has prior runs)')
            lines.append('next_step=use Resume Existing Model or Prepare New Model Directory')
        else:
            lines.append('action=fresh training launch (creates run_0001 for new model directory)')
    else:
        source = preview.get('source_run_info')
        if source is None:
            lines.append('action=blocked (no resumable source run found)')
            lines.append('next_step=select a model with at least one resumable run, then Get Info')
        else:
            required_remaining = int(preview.get('required_remaining', 0))
            add_epochs = int(additional_epochs_input.value)
            total_additional = required_remaining + add_epochs
            lines.append(f"source_run_id={source['run_id']}")
            lines.append(f'required_remaining={required_remaining}')
            lines.append(f'add_epochs={add_epochs}')
            lines.append(f'total_additional_epochs={total_additional}')
    return "\n".join(lines)


def _update_preview_paths(show_error: bool = False) -> dict | None:
    topology_id = str(topology_id_select.value or '').strip() or TOPOLOGY_ID_DEFAULT
    topology_variant = str(topology_variant_select.value or '').strip() or MODEL_NAME_DEFAULT
    model_name = topology_variant
    model_directory = model_directory_input.value.strip()
    mode = workflow_toggle.value

    if not model_directory:
        run_id_preview.value = ''
        source_run_preview.value = ''
        derived_log_path.value = ''
        launch_plan_output.value = '[model directory is empty]'
        session_suggestion.value = '<em>Suggested session name: <code>rb-&lt;workflow&gt;-&lt;run_id&gt;</code></em>'
        return None

    try:
        next_run_id = control.preview_next_run_id(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
        run_id_preview.value = next_run_id

        log_path = control.build_log_path(
            run_id=next_run_id,
            models_root=MODELS_ROOT,
            model_directory=model_directory,
            log_filename=LOG_FILENAME,
        )
        derived_log_path.value = str(log_path)

        source_info = None
        required_remaining = 0
        source_run_preview.value = ''
        if mode == WORKFLOW_RESUME:
            source_info = _model_state.get('latest_resumable')
            if source_info is not None:
                source_run_preview.value = str(source_info['run_id'])
                total_epochs = source_info.get('total_epochs')
                completed_epochs = source_info.get('completed_epochs')
                if isinstance(total_epochs, int) and isinstance(completed_epochs, int):
                    required_remaining = max(int(total_epochs) - int(completed_epochs), 0)

                source_topology_id = str(source_info.get('topology_id', '')).strip()
                if source_topology_id:
                    topology_id = source_topology_id
                source_topology_variant = str(
                    source_info.get('topology_variant')
                    or source_info.get('architecture_variant')
                    or ''
                ).strip()
                if source_topology_variant:
                    topology_variant = source_topology_variant
                    model_name = source_topology_variant

        session_base = model_name if model_name else 'model'
        if mode == WORKFLOW_NEW:
            suggested_session = f'rb-new-{session_base}-{next_run_id}'
        elif source_info is not None:
            suggested_session = f"rb-resume-{source_info['run_id']}-to-{next_run_id}"
        else:
            suggested_session = f'rb-resume-{session_base}-{next_run_id}'

        session_suggestion.value = f"<em>Suggested session name: <code>{html.escape(suggested_session)}</code></em>"
        _sync_default_session_name(suggested_session)

        preview = {
            'mode': mode,
            'model_name': model_name,
            'topology_id': topology_id,
            'topology_variant': topology_variant,
            'model_directory': model_directory,
            'run_id': next_run_id,
            'log_path': str(log_path),
            'suggested_session': suggested_session,
            'existing_rows': len(_model_state.get('rows', [])),
            'source_run_info': source_info,
            'required_remaining': required_remaining,
        }
        launch_plan_output.value = _build_launch_plan_text(preview)
        return preview
    except Exception as exc:
        run_id_preview.value = ''
        source_run_preview.value = ''
        derived_log_path.value = ''
        launch_plan_output.value = '[launch plan unavailable]'
        if show_error:
            _set_status(f'Path derivation failed: {exc}', is_error=True)
        return None


def _refresh_sessions(*, preferred: str | None = None) -> None:
    try:
        sessions = control.list_sessions()
    except Exception as exc:
        session_select.options = []
        session_select.value = None
        _set_status(f'Session refresh failed: {exc}', is_error=True)
        return

    current = preferred if preferred is not None else session_select.value
    session_select.options = sessions
    if sessions:
        if current in sessions:
            session_select.value = current
        else:
            session_select.value = sessions[0]
    else:
        session_select.value = None


def _resolve_log_path_from_selected_session() -> str | None:
    selected = session_select.value or session_name_input.value.strip()
    if not selected:
        preview = _update_preview_paths(show_error=False)
        return None if preview is None else preview['log_path']

    try:
        resolved = control.resolve_session_run(
            session_name=selected,
            models_root=MODELS_ROOT,
        )
    except Exception as exc:
        _set_status(f'Session lookup failed: {exc}', is_error=True)
        return None

    if resolved is None:
        return None

    model_directory_input.value = resolved['model_directory']
    if resolved['model_directory'] in model_select.options:
        model_select.value = resolved['model_directory']
    run_id_preview.value = resolved['run_id']
    derived_log_path.value = resolved['log_path']
    return str(resolved['log_path'])


def _resolve_run_dir_for_summary() -> Path | None:
    selected = session_select.value or session_name_input.value.strip()
    if selected:
        try:
            resolved = control.resolve_session_run(
                session_name=selected,
                models_root=MODELS_ROOT,
            )
        except Exception as exc:
            _set_status(f'Session lookup failed: {exc}', is_error=True)
            return None
        if resolved is not None:
            try:
                return control.build_run_dir(
                    run_id=resolved['run_id'],
                    models_root=MODELS_ROOT,
                    model_directory=resolved['model_directory'],
                ).resolve()
            except Exception as exc:
                _set_status(f'Run directory resolution failed: {exc}', is_error=True)
                return None

    model_directory = model_directory_input.value.strip()
    run_id = run_id_preview.value.strip()
    if not model_directory or not run_id:
        return None

    try:
        return control.build_run_dir(
            run_id=run_id,
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        ).resolve()
    except Exception:
        return None


def _refresh_epoch_summary(*_) -> None:
    run_dir = _resolve_run_dir_for_summary()
    _set_epoch_summary_text(control.read_epoch_summary(run_dir))


def _refresh_log(*_) -> None:
    _refresh_epoch_summary()
    selected_session = session_select.value or session_name_input.value.strip()
    log_path = _resolve_log_path_from_selected_session()
    if log_path is None:
        if selected_session:
            log_output.value = f'[no registered run for session: {selected_session}]'
        else:
            log_output.value = '[no session selected]'
        return

    try:
        tail_lines = int(log_tail_lines_input.value)
        log_output.value = control.read_log_tail(log_path, max_lines_or_chars=tail_lines)
    except Exception as exc:
        log_output.value = ''
        _set_status(f'Log refresh failed: {exc}', is_error=True)


def _stop_auto_refresh() -> None:
    stop_event = _auto_refresh_state.get('stop_event')
    thread = _auto_refresh_state.get('thread')
    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive():
        thread.join(timeout=1.0)
    _auto_refresh_state['thread'] = None
    _auto_refresh_state['stop_event'] = None


def _auto_refresh_loop(stop_event: threading.Event) -> None:
    while not stop_event.is_set():
        try:
            _refresh_log()
        except Exception as exc:
            _set_status(f'Auto refresh failed: {exc}', is_error=True)
        interval = max(0.5, float(poll_interval_input.value))
        stop_event.wait(interval)


def _on_auto_refresh_toggle(change):
    enabled = bool(change['new'])
    if enabled:
        _stop_auto_refresh()
        stop_event = threading.Event()
        thread = threading.Thread(target=_auto_refresh_loop, args=(stop_event,), daemon=True)
        _auto_refresh_state['thread'] = thread
        _auto_refresh_state['stop_event'] = stop_event
        thread.start()
        _set_status('Auto refresh enabled.')
    else:
        _stop_auto_refresh()
        _set_status('Auto refresh disabled.')


def _sync_mode_controls() -> None:
    is_new = workflow_toggle.value == WORKFLOW_NEW
    additional_epochs_input.disabled = is_new
    if is_new:
        additional_epochs_input.value = 0


def _prepare_new_model_directory(*, update_status: bool = True) -> None:
    try:
        candidate = _suggest_unique_model_directory()
    except Exception as exc:
        _set_status(f'Could not prepare new model directory: {exc}', is_error=True)
        return

    model_directory_input.value = candidate
    _refresh_models()
    _refresh_model_state(show_error=False)
    preview = _update_preview_paths(show_error=False)
    model_info_output.value = _build_model_info_text(candidate)
    if update_status:
        next_run = preview['run_id'] if preview is not None else 'run_0001'
        _set_status(
            f'Prepared new model directory {candidate}. Next step: review settings and launch {next_run}.'
        )


def _on_get_info_clicked(_):
    model_directory = model_directory_input.value.strip()
    if not model_directory:
        model_info_output.value = 'Model directory is empty.'
        _set_status('Model directory is required for Get Info.', is_error=True)
        return

    _refresh_model_state(show_error=True)
    _update_preview_paths(show_error=False)
    model_info_output.value = _build_model_info_text(model_directory)
    _set_status(f'Loaded model info for {model_directory}.')


def _validate_new_model_launch(preview: dict) -> None:
    model_directory = str(preview['model_directory'])
    rows = list(_model_state.get('rows', []))
    if rows:
        raise ValueError(
            f'Model directory {model_directory} already has {len(rows)} run(s). '
            'Use Resume Existing Model, or click Prepare New Model Directory for a fresh timestamped model.'
        )


def _on_launch_clicked(_):
    preview = _update_preview_paths(show_error=True)
    if preview is None:
        return

    session_name = session_name_input.value.strip()
    if not session_name:
        _set_status(
            f"Session name is required. Suggested: {preview['suggested_session']}",
            is_error=True,
        )
        return

    if control.session_exists(session_name):
        _set_status(f'Duplicate session name refused: {session_name}', is_error=True)
        return

    mode = preview['mode']

    try:
        model_arch_variant = preview['model_name']
        topology_id = str(preview.get('topology_id', TOPOLOGY_ID_DEFAULT)).strip() or TOPOLOGY_ID_DEFAULT
        topology_variant = str(preview.get('topology_variant', model_arch_variant)).strip() or model_arch_variant
        if mode == WORKFLOW_NEW:
            _validate_new_model_launch(preview)
            parent_run_id = RESERVED_PARENT_RUN_ID
            additional_for_resume = None
            source_info = None
        else:
            source_info = preview.get('source_run_info')
            if source_info is None:
                raise ValueError(
                    'No resumable source run found in selected model directory. '
                    'Use Get Info to inspect runs or switch to Train New Model.'
                )

            parent_run_id = str(source_info['run_id'])
            source_variant = str(source_info.get('topology_variant', '')).strip()
            if not source_variant:
                source_variant = str(source_info.get('architecture_variant', '')).strip()
            if source_variant:
                model_arch_variant = source_variant
                topology_variant = source_variant

            source_topology_id = str(source_info.get('topology_id', '')).strip()
            if source_topology_id:
                topology_id = source_topology_id

            required_remaining = int(preview.get('required_remaining', 0))
            add_epochs = int(additional_epochs_input.value)
            if add_epochs < 0:
                raise ValueError(f'Add Epochs must be >= 0; got {add_epochs}')

            additional_for_resume = required_remaining + add_epochs
            if additional_for_resume <= 0:
                total_epochs = source_info.get('total_epochs')
                completed_epochs = source_info.get('completed_epochs')
                raise ValueError(
                    f"Selected source run {source_info['run_id']} appears complete "
                    f"({completed_epochs}/{total_epochs}). Set Add Epochs > 0 or train a new model."
                )

        reservation = control.reserve_run(
            models_root=MODELS_ROOT,
            model_directory=preview['model_directory'],
            model_name=model_arch_variant,
            topology_id=topology_id,
            topology_variant=topology_variant,
            session_name=session_name,
            primary_variable_changed=primary_variable_input.value.strip(),
            notes=notes_input.value.strip(),
            parent_run_id=parent_run_id,
        )

        run_id_preview.value = reservation['run_id']
        run_register_path.value = reservation['run_register_path']
        derived_log_path.value = reservation['log_path']

        if mode == WORKFLOW_NEW:
            command = control.build_launch_command(
                run_id=reservation['run_id'],
                model_directory=preview['model_directory'],
                model_architecture_variant=topology_variant,
                topology_id=topology_id,
                topology_params_json=topology_params_json_input.value.strip() or '{}',
                python_executable=PYTHON_EXECUTABLE,
                training_module=TRAINING_MODULE,
                training_data_root=TRAINING_DATA_ROOT,
                validation_data_root=VALIDATION_DATA_ROOT,
                output_root=MODELS_ROOT,
                seed=int(seed_input.value),
                batch_size=int(batch_size_input.value),
                epochs=int(epochs_input.value),
                learning_rate=float(learning_rate_input.value),
                weight_decay=float(weight_decay_input.value),
                early_stopping_patience=int(early_stop_input.value),
                enable_lr_scheduler=bool(enable_lr_scheduler_toggle.value),
                change_note=change_note_input.value.strip() or 'tmux control-panel launch',
            )
            launch_mode_message = (
                f"Launched NEW model training: {preview['model_directory']} / {reservation['run_id']} "
                f'(session {session_name}).'
            )
        else:
            command = resume_control.build_resume_launch_command(
                run_id=reservation['run_id'],
                model_directory=preview['model_directory'],
                source_run_dir=source_info['run_dir'],
                additional_epochs=int(additional_for_resume),
                python_executable=PYTHON_EXECUTABLE,
                training_module=TRAINING_MODULE,
                output_root=MODELS_ROOT,
                change_note=change_note_input.value.strip() or 'resume launch',
            )
            launch_mode_message = (
                f"Launched RESUME: {source_info['run_id']} -> {reservation['run_id']} "
                f"(additional_epochs={additional_for_resume}; session {session_name})."
            )

        control.launch_session(
            session_name=session_name,
            command=command,
            log_path=reservation['log_path'],
            working_directory=REPO_ROOT,
        )

        _refresh_models()
        _refresh_model_state(show_error=False)
        _refresh_sessions(preferred=session_name)
        _update_preview_paths(show_error=False)
        model_info_output.value = _build_model_info_text(preview['model_directory'])
        _refresh_log()
        _set_status(launch_mode_message)
    except Exception as exc:
        _set_status(f'Launch failed: {exc}', is_error=True)


def _on_end_session_clicked(_):
    selected_session = session_select.value
    if not selected_session:
        _set_status('Select a session to end.', is_error=True)
        return
    try:
        ended = control.end_session(selected_session)
        if ended:
            _set_status(f'Ended session {selected_session}.')
        else:
            _set_status(f'Session not found: {selected_session}', is_error=True)
        _refresh_sessions()
        _refresh_log()
    except Exception as exc:
        _set_status(f'End session failed: {exc}', is_error=True)


def _on_session_changed(change):
    selected = change['new']
    if selected:
        session_name_input.value = selected
    _refresh_log()


def _on_workflow_changed(change):
    _ = change
    _sync_mode_controls()
    _set_workflow_help()

    if workflow_toggle.value == WORKFLOW_NEW:
        _prepare_new_model_directory(update_status=False)
    else:
        selected = model_select.value
        if selected:
            model_directory_input.value = str(selected)
        _refresh_model_state(show_error=False)
        _update_preview_paths(show_error=False)


def _on_model_selected(change):
    selected = change['new']
    if selected:
        model_directory_input.value = str(selected)
    _refresh_model_state(show_error=False)
    _update_preview_paths(show_error=False)


def _on_model_directory_changed(_):
    _refresh_model_state(show_error=False)
    _update_preview_paths(show_error=False)


def _on_topology_id_changed(change):
    _ = change
    _sync_topology_variant_options()
    _update_preview_paths(show_error=False)


def _on_preview_inputs_changed(_):
    _set_topology_help()
    _update_preview_paths(show_error=False)


launch_button.on_click(_on_launch_clicked)
end_session_button.on_click(_on_end_session_clicked)
refresh_sessions_button.on_click(lambda _: _refresh_sessions())
refresh_log_button.on_click(_refresh_log)
refresh_models_button.on_click(_refresh_models)
refresh_topologies_button.on_click(lambda _: _refresh_topology_options(update_status=True))
get_info_button.on_click(_on_get_info_clicked)
prepare_new_model_button.on_click(lambda _: _prepare_new_model_directory(update_status=True))

session_select.observe(_on_session_changed, names='value')
workflow_toggle.observe(_on_workflow_changed, names='value')
model_select.observe(_on_model_selected, names='value')
model_directory_input.observe(_on_model_directory_changed, names='value')
topology_id_select.observe(_on_topology_id_changed, names='value')
topology_variant_select.observe(_on_preview_inputs_changed, names='value')
topology_params_json_input.observe(_on_preview_inputs_changed, names='value')
auto_refresh_toggle.observe(_on_auto_refresh_toggle, names='value')

_set_workflow_help()
_sync_mode_controls()
_refresh_topology_options(update_status=False)
_refresh_models()
_prepare_new_model_directory(update_status=False)
_refresh_epoch_summary()
_refresh_sessions()
_set_status('Control panel ready. Default workflow is Train New Model.')

display(widgets.VBox([
    status_area,
    workflow_help,
    widgets.HBox([workflow_toggle, prepare_new_model_button]),
    widgets.HBox([topology_id_select, topology_variant_select, refresh_topologies_button]),
    topology_note_input,
    topology_help,
    model_directory_input,
    widgets.HBox([model_select, refresh_models_button, get_info_button]),
    model_info_output,
    session_name_input,
    session_suggestion,
    widgets.HBox([seed_input, batch_size_input, epochs_input, additional_epochs_input]),
    widgets.HBox([learning_rate_input, weight_decay_input, early_stop_input, enable_lr_scheduler_toggle]),
    topology_params_json_input,
    change_note_input,
    primary_variable_input,
    notes_input,
    widgets.HBox([run_id_preview, source_run_preview]),
    run_register_path,
    derived_log_path,
    launch_plan_output,
    widgets.HBox([launch_button, refresh_sessions_button, end_session_button]),
    session_select,
    widgets.HBox([refresh_log_button, log_tail_lines_input, poll_interval_input, auto_refresh_toggle]),
    epoch_summary_output,
    log_output,
]))


## Final Verdict

`READY_FOR_LAUNCH` when the panel clearly supports:
- `Train New Model` as the default path with a prepared fresh timestamped model directory.
- `Resume Existing Model` from the latest resumable run in a selected model directory.
- `Get Info` highlighting completed vs total epochs before launch.
